# PML · Lecture 11 — Clustering & K-means

Unsupervised learning through its workhorse algorithm. We build **K-means** from scratch, *watch* it minimise the distortion $J$ by alternating two steps, and *see* exactly where and why it breaks — setting up the soft, probabilistic version (a Gaussian mixture) in Lecture 13.

Everything here is pure **`numpy` + `matplotlib`** (datasets from `scikit-learn`), no setup and no API key — run each cell top to bottom. We mirror the reading: the clustering problem, the objective $J$, Lloyd's algorithm, the by-hand worked examples, k-means++, the four limitations, and a **Practice** section that lets you *check your pen-and-paper answers* to the exercises.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (6.2, 4.6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
PALETTE = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#ff7f0e', '#17becf', '#8c564b', '#e377c2']
print('ready — numpy', np.__version__)

## 1. The clustering problem

Every model so far was **supervised** — inputs $x$ *and* targets $y$. Now we drop the targets: we're handed points $x_1,\dots,x_N$ with **no labels** and asked *do they fall into natural groups?* That's **clustering**, the canonical **unsupervised** task. The goal: split the points into $K$ groups so points in a group are **close** and points in different groups are **far**. Each cluster is summarised by a **centroid** $\mu_k$ — its prototype (drawn as a big ✕). When the blobs are tidy and well-separated, the answer is the one your eye already sees:

In [ ]:
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=0.9, random_state=42)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].scatter(X[:, 0], X[:, 1], s=12, c='0.5')
ax[0].set_title('what the algorithm gets: no labels')
for k in range(3):
    ax[1].scatter(*X[y_true == k].T, s=12, c=PALETTE[k])
    ax[1].scatter(*X[y_true == k].mean(0), marker='X', s=260, c=PALETTE[k],
                  edgecolor='k', linewidth=1.5)
ax[1].set_title('the grouping your eye sees (✕ = centroid)')
plt.tight_layout(); plt.show()

Clustering is everywhere a label is missing or expensive — customer segmentation, colour quantization, topic grouping, anomaly detection, vector quantization. **The catch:** with no labels there's no single right answer to check against, so we optimise a *surrogate* objective that encodes "good grouping." For K-means that objective is the distortion $J$.

## 2. The K-means objective $J$

Each cluster $k$ is one centroid $\mu_k$. Membership is a **hard binary assignment** $r_{nk}\in\{0,1\}$ with $\sum_k r_{nk}=1$ (each point in exactly one cluster). The **distortion** is the total squared distance from each point to *its* centroid:

$$J = \sum_n \sum_k r_{nk}\,\lVert x_n - \mu_k \rVert^2$$

Small $J$ = tight, well-separated clusters. K-means chooses the $r_{nk}$ **and** the $\mu_k$ that together minimise $J$. In code the objective is one line:

In [ ]:
def distortion(X, r, mu):
    """Total squared distance from each point to its assigned centroid."""
    return float(sum(((X[r == k] - mu[k]) ** 2).sum() for k in range(len(mu))))

# a deliberately-bad hand assignment vs the sensible one, to show J *measures* quality
mu_demo = np.array([X[y_true == k].mean(0) for k in range(3)])
r_good = y_true
r_bad = (y_true + 1) % 3                                   # shuffle every point to a wrong centroid
print(f"J (sensible assignment): {distortion(X, r_good, mu_demo):8.1f}")
print(f"J (scrambled assignment):{distortion(X, r_bad,  mu_demo):8.1f}   <- much larger")

$J$ couples two kinds of unknown: the discrete $r_{nk}$ (*which* cluster) and the continuous $\mu_k$ (*where* the centroids sit). Minimising over both at once is NP-hard — so the algorithm solves one at a time.

## 3. Lloyd's algorithm

Minimise $J$ by **alternating** two steps, each an *exact* minimiser for one set of variables with the other fixed:

- **Assignment** (fix $\mu$, solve $r$): each point → its **nearest** centroid.
- **Update** (fix $r$, solve $\mu$): each centroid → the **mean** of its assigned points.

Repeat until assignments stop changing. Each step can only **lower $J$ or leave it unchanged** (it's coordinate descent), so $J$ is monotonically non-increasing and converges — but only to a **local** optimum that depends on the start. Here it is from scratch, straight from the reading:

In [ ]:
def kmeans(X, K, iters=50, seed=0, mu_init=None, return_hist=False):
    rng = np.random.default_rng(seed)
    mu = X[rng.choice(len(X), K, replace=False)].astype(float) if mu_init is None \
         else np.asarray(mu_init, float)
    hist = []
    for _ in range(iters):
        d = ((X[:, None, :] - mu[None, :, :]) ** 2).sum(-1)   # (N, K) squared distances
        r = d.argmin(1)                                        # ASSIGN: nearest centroid
        hist.append(distortion(X, r, mu))
        new_mu = np.array([X[r == k].mean(0) if np.any(r == k) else mu[k]  # UPDATE: mean
                           for k in range(K)])
        if np.allclose(new_mu, mu):
            break
        mu = new_mu
    r = ((X[:, None, :] - mu[None, :, :]) ** 2).sum(-1).argmin(1)   # final r vs final centroids
    J = distortion(X, r, mu)
    return (r, mu, J, hist) if return_hist else (r, mu, J)

# A single random start can land in a bad local optimum (that's the whole point of §3d),
# so keep the best of a few restarts — the practical fix the reading recommends.
r, mu, J = min((kmeans(X, K=3, seed=s) for s in range(8)), key=lambda t: t[2])
for k in range(3):
    plt.scatter(*X[r == k].T, s=12, c=PALETTE[k])
    plt.scatter(*mu[k], marker='X', s=260, c=PALETTE[k], edgecolor='k', linewidth=1.5)
plt.title(f'K-means, K=3, best of 8 restarts   (distortion J = {J:.1f})')
plt.show()

### Watch $J$ fall every iteration
Coordinate descent means the distortion can only go **down**. Plot it per sweep:

In [ ]:
best_seed = min(range(8), key=lambda s: kmeans(X, K=3, seed=s)[2])   # same good run as above
_, _, _, hist = kmeans(X, K=3, seed=best_seed, return_hist=True)
plt.plot(range(len(hist)), hist, 'o-')                              # hist = J at each assignment step
plt.xlabel('iteration'); plt.ylabel('distortion J'); plt.title('J is monotonically non-increasing')
plt.show()
print('J per iteration:', [round(h, 1) for h in hist])

### Work it through — one full iteration by hand (1-D)
Six points on a line $x = 1,2,3,10,11,12$, $K=2$, start $\mu_1=2,\ \mu_2=9$. One full iteration (assign → update). The reading gets $\mu=(2,11)$ and $J=4$; reproduce it and print the assignment table:

In [ ]:
x1d = np.array([1, 2, 3, 10, 11, 12.])[:, None]
mu0 = np.array([[2.], [9.]])

d = ((x1d[:, None, :] - mu0[None, :, :]) ** 2).sum(-1)      # squared dist to each centroid
r = d.argmin(1)
print(" x  | (x-2)^2 (x-9)^2 | cluster")
for xi, (d1, d2), ri in zip(x1d.ravel(), d, r):
    print(f"{xi:3.0f} | {d1:6.0f} {d2:7.0f} | {ri + 1}")
mu_new = np.array([x1d[r == k].mean(0) for k in range(2)])
print("\nupdated centroids:", mu_new.ravel(), "-> expect [2, 11]")
print("distortion J     :", distortion(x1d, r, mu_new), "-> expect 4.0")

### Work it through — the mean minimises squared distance (*why* it's called K-**means**)
For one cluster the cost $C(\mu)=\sum_n (x_n-\mu)^2$ is minimised where $dC/d\mu=0$, which gives the **mean**. For $\{1,2,3\}$ that's $\mu=2$. Plot the parabola and mark the minimum:

In [ ]:
pts = np.array([1., 2., 3.])
grid = np.linspace(0, 4, 200)
C = ((pts[None, :] - grid[:, None]) ** 2).sum(1)
plt.plot(grid, C)
plt.axvline(pts.mean(), color='#d62728', ls='--', label=f'mean = {pts.mean():.0f} (the minimiser)')
plt.scatter([pts.mean()], [((pts - pts.mean()) ** 2).sum()], color='#d62728', zorder=5)
plt.xlabel('centroid μ'); plt.ylabel('cost C(μ)'); plt.legend(); plt.title('cost is minimised at the mean')
plt.show()
# derivative check: dC/dμ = -2 Σ(x_n - μ) = 0  ->  μ = mean
print("derivative at μ=2:", -2 * (pts - 2).sum(), "(zero -> μ=2 is the stationary point)")

### Work it through — $J$ drops on the update step (2-D)
Points $a{=}(0,0),b{=}(0,2),c{=}(5,0),d{=}(5,2)$, $K=2$, start $\mu_1{=}(0,0),\mu_2{=}(1,2)$. Assign, then update, and compare $J$ before/after — the reading gets $37 \to \approx 19.3$ **without changing any assignment**:

In [ ]:
X2 = np.array([[0, 0], [0, 2], [5, 0], [5, 2.]])
mu = np.array([[0, 0], [1, 2.]])
r = ((X2[:, None, :] - mu[None, :, :]) ** 2).sum(-1).argmin(1)
J_before = distortion(X2, r, mu)
mu_new = np.array([X2[r == k].mean(0) for k in range(2)])   # update only; keep same r
J_after = distortion(X2, r, mu_new)
print(f"assignments: {r}  (cluster of a,b,c,d)")
print(f"J before update: {J_before:.1f}   (expect 37)")
print(f"μ2 after update: {mu_new[1].round(3)}   (expect [3.333 1.333])")
print(f"J after update : {J_after:.2f}  (expect ~19.33)  -> dropped, same assignments")

### Better starts — k-means++
Random init can bunch the seeds and land in a poor local optimum. **k-means++** spreads them: first centroid random, then each next one chosen with probability **∝ squared distance** to the nearest seed already chosen. Distant points are far likelier — seeds rarely clump. It's scikit-learn's default. Implement it and reproduce the reading's $\{0,1,10\}$ probabilities:

In [ ]:
def kpp_init(X, K, rng):
    mu = [X[rng.integers(len(X))]]                          # first centroid: uniform random
    for _ in range(1, K):
        d2 = np.min([((X - c) ** 2).sum(1) for c in mu], axis=0)  # dist^2 to NEAREST chosen
        s = d2.sum()
        p = d2 / s if s > 0 else np.ones(len(X)) / len(X)   # all-duplicate -> uniform (no NaN)
        mu.append(X[rng.choice(len(X), p=p)])               # pick ∝ d^2
    return np.array(mu)

# the reading's tiny example: first centroid = 0, remaining points {1, 10}
pts = np.array([1., 10.]); d2 = pts ** 2
print("k-means++ pick probabilities after choosing x=0:")
for xi, p in zip(pts, d2 / d2.sum()):
    print(f"  P(x={xi:>2.0f}) = {p:.4f}")
print("  -> the far point x=10 is chosen ~99% of the time: one seed per natural group")

## 4. Limitations — where K-means breaks

| Limitation | What goes wrong |
|---|---|
| **You must choose $K$** | it's an input, not learned — use the *elbow method* or silhouette |
| **Sensitive to init** | non-convex $J$ → different starts → different local optima |
| **Sensitive to feature scale** | large-range features dominate $\lVert\cdot\rVert^2$ — standardise first |
| **Assumes spherical, equal-size clusters** | the squared-distance objective wants round, similar blobs |
| **Hard assignments** | a point exactly between two clusters is forced fully into one — no "60% A, 40% B" or uncertainty (exactly what the GMM's soft responsibilities fix) |

### Choosing $K$: the elbow method
$J$ only falls as $K$ grows, so the signal isn't the minimum but the **bend** — the $K$ past which extra clusters stop buying much. On our 3-blob data the elbow is at $K=3$:

In [ ]:
Ks = range(1, 9)
Js = [min(kmeans(X, K=k, seed=s)[2] for s in range(5)) for k in Ks]   # best-of-5 per K
plt.plot(list(Ks), Js, 'o-')
plt.axvline(3, color='#d62728', ls='--', label='elbow at K=3')
plt.xlabel('number of clusters K'); plt.ylabel('distortion J'); plt.legend()
plt.title('elbow method: the bend, not the minimum'); plt.show()

### Sensitive to initialisation
Run plain K-means from many random seeds on a harder 6-blob set: many runs find the best value, but a chunk land in worse local optima — the **high-$J$ tail**. k-means++ starts reach the same best minimum while **trimming that tail** (fewer bad runs):

In [ ]:
Xh, _ = make_blobs(n_samples=500, centers=6, cluster_std=1.1, random_state=7)
J_rand = [kmeans(Xh, K=6, seed=s)[2] for s in range(40)]
J_kpp = [kmeans(Xh, K=6, seed=s, mu_init=kpp_init(Xh, 6, np.random.default_rng(s)))[2]
          for s in range(40)]
plt.hist(J_rand, bins=15, alpha=0.6, label=f'random init (spread; min {min(J_rand):.0f})')
plt.hist(J_kpp,  bins=15, alpha=0.6, label=f'k-means++ (min {min(J_kpp):.0f})')
plt.xlabel('final distortion J'); plt.ylabel('count of seeds'); plt.legend()
plt.title('init matters: random init scatters into local optima'); plt.show()
print(f"random init  J: min {min(J_rand):.0f}, max {max(J_rand):.0f}")
print(f"k-means++    J: min {min(J_kpp):.0f}, max {max(J_kpp):.0f}")

### Sensitive to feature scale
$\lVert\cdot\rVert^2$ lets a wide-range feature dominate the distance. Here two groups are separated **only in feature 0**; feature 1 is uninformative noise — but on a **20× larger scale**. Raw K-means clusters by the big, meaningless axis (wrong); **standardising** to unit variance first restores the true split. Both panels are drawn in the *same original coordinates* so you can compare the groupings directly:

In [ ]:
def best_of(Xd, K, restarts=8):                            # keep the lowest-J random restart
    return min((kmeans(Xd, K, seed=s) for s in range(restarts)), key=lambda t: t[2])

n = 200
x_info = np.concatenate([rng.normal(-3, 0.5, n), rng.normal(3, 0.5, n)])   # separates the 2 groups
y_noise = rng.normal(0, 1.0, 2 * n)                        # feature 1: pure noise, no group info
X_orig = np.column_stack([x_info, y_noise]); y_fs = np.repeat([0, 1], n)
Xf = X_orig.copy(); Xf[:, 1] *= 20                         # feature 1 now dominates ||·||^2

r_raw, _, _ = best_of(Xf, 2)                               # cluster the raw (unscaled) data
Xz = (Xf - Xf.mean(0)) / Xf.std(0)                         # standardise: zero mean, unit variance
r_std, _, _ = best_of(Xz, 2)

purity = lambda r: sum(np.bincount(y_fs[r == k], minlength=2).max() for k in range(2)) / len(y_fs)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for k in range(2):                                         # plot both in ORIGINAL coords -> comparable
    ax[0].scatter(*X_orig[r_raw == k].T, s=10, c=PALETTE[k])
    ax[1].scatter(*X_orig[r_std == k].T, s=10, c=PALETTE[k])
ax[0].set_title(f'raw: feature-1 scale dominates — wrong (purity {purity(r_raw):.2f})')
ax[1].set_title(f'standardised first: splits by feature 0 (purity {purity(r_std):.2f})')
for a in ax:
    a.set_xlabel('feature 0'); a.set_ylabel('feature 1')
plt.tight_layout(); plt.show()

### Assumes round, equal-size blobs — the two-moons failure
Every boundary K-means can draw is a **straight line** halfway between two centroids, so it can't follow a curved cluster. On two interleaving crescents it slices straight *across* both moons instead of separating them:

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.06, random_state=0)
r_m, mu_m, _ = kmeans(Xm, K=2, seed=0)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for k in range(2):
    ax[0].scatter(*Xm[ym == k].T, s=12, c=PALETTE[k])
ax[0].set_title('true structure: two crescents')
for k in range(2):
    ax[1].scatter(*Xm[r_m == k].T, s=12, c=PALETTE[k])
    ax[1].scatter(*mu_m[k], marker='X', s=240, c=PALETTE[k], edgecolor='k', linewidth=1.5)
ax[1].set_title('K-means K=2: slices across, not along')
plt.tight_layout(); plt.show()

### Foreshadowing: the probabilistic version (Lecture 13)
Each limitation loosens if we treat clustering as a **probabilistic model**. A **Gaussian Mixture Model** says the data is generated by $K$ Gaussian blobs, each with its own mean *and* covariance, fit by the **EM algorithm** — giving **soft assignments** (a probability per cluster, replacing $r_{nk}$) and **non-spherical** clusters (a full covariance per component). In fact K-means is the limiting case of a GMM with equal, spherical, shrinking-variance components and hard responsibilities. Keep the assign↔update loop in mind when EM arrives.

## Practice — check your by-hand answers

The reading's exercises are meant to be done **pen-and-paper** to *feel* the assign↔update loop. Use these cells to **verify** each answer once you've worked it out. First, two helpers that do exactly one iteration and run to convergence, printing what changed:

In [ ]:
def one_iteration(X, mu):
    """One assign→update sweep. Returns (assignments, updated centroids, J_after)."""
    mu = np.asarray(mu, float)
    r = ((X[:, None, :] - mu[None, :, :]) ** 2).sum(-1).argmin(1)
    mu_new = np.array([X[r == k].mean(0) if np.any(r == k) else mu[k] for k in range(len(mu))])
    return r, mu_new, distortion(X, r, mu_new)

def to_line(x):                      # helper: shape a 1-D list into an (N,1) array
    return np.asarray(x, float)[:, None]
print('helpers ready: one_iteration(X, mu), to_line([...]), distortion(X, r, mu), kmeans(...)')

**Exercises 1–4 (1-D).** Points $x=0,1,5,6$, $K=2$, start $\mu_1=0,\mu_2=1$. Predict the assignment, the updated centroids, $J$, and the converged result — then run:

In [ ]:
X14 = to_line([0, 1, 5, 6])
# TODO first predict on paper: which cluster each point joins, then μ after one update
r, mu1, J1 = one_iteration(X14, [[0], [1]])
print("Ex1/2 assignment:", r, "| updated μ:", mu1.ravel(), " (expect [0, 4])")
print("Ex3  distortion J:", J1, " (expect 14)")
r, mu_c, J_c = kmeans(X14, K=2, mu_init=[[0], [1]])
print("Ex4  converged  μ:", mu_c.ravel(), "| J:", J_c, " (expect [0.5, 5.5], J = 1)")

**Exercise 5 — the mean is the optimal centroid.** Cluster $\{2,6,7\}$: the cost $C(\mu)=(2-\mu)^2+(6-\mu)^2+(7-\mu)^2$ is minimised at the mean. Confirm the derivative vanishes there:

In [ ]:
pts = np.array([2., 6., 7.])
mu_star = pts.mean()
print("mean (minimiser):", mu_star, " (expect 5)")
print("dC/dμ at the mean:", -2 * (pts - mu_star).sum(), " (zero -> it's the stationary point)")
print("2nd derivative   :", 2 * len(pts), " (> 0 -> a minimum)")

**Exercises 6 & 7 — each step lowers $J$.** Ex6: nearest-centroid assignment beats a wrong one. Ex7: moving a centroid to the mean lowers the cluster cost. Predict which is smaller, then check:

In [ ]:
# Ex6: x=0,2,9 ; centroids μ1=1, μ2=8 — nearest vs a deliberately-wrong assignment
X6 = to_line([0, 2, 9]); mu6 = np.array([[1.], [8.]])
r_near = ((X6[:, None, :] - mu6[None, :, :]) ** 2).sum(-1).argmin(1)
r_wrong = np.array([0, 1, 1])                         # send x=2 to the far centroid
print("Ex6 J nearest:", distortion(X6, r_near, mu6), "| J wrong:", distortion(X6, r_wrong, mu6),
      " (expect 3 vs 38)")

# Ex7: cluster {1,3,8}; centroid at μ=1 vs at the mean μ=4
c = np.array([1., 3., 8.])
print("Ex7 cost at μ=1:", ((c - 1) ** 2).sum(), "| at mean μ=4:", ((c - c.mean()) ** 2).sum(),
      "| dropped by", int(((c - 1) ** 2).sum() - ((c - c.mean()) ** 2).sum()), " (expect 53, 26, 27)")

**Exercise 8 (2-D) & Exercise 10 (bad init).** Ex8: points $(1,1),(1,2),(8,1),(8,0)$, start $\mu_1{=}(0,0),\mu_2{=}(9,0)$. Ex10: $x=1,2,9,10$ with a cramped start $\mu_1{=}1,\mu_2{=}2$ — does it recover the obvious split?

In [ ]:
X8 = np.array([[1, 1], [1, 2], [8, 1], [8, 0.]])
r8, mu8, J8 = kmeans(X8, K=2, mu_init=[[0, 0], [9, 0]])
print("Ex8  μ1:", mu8[0], "μ2:", mu8[1], "| J:", J8, " (expect (1,1.5),(8,0.5), J = 1)")

X10 = to_line([1, 2, 9, 10])
r10, mu10, J10 = kmeans(X10, K=2, mu_init=[[1], [2]])
print("Ex10 converged μ:", mu10.ravel(), "| J:", J10,
      " (expect [1.5, 9.5], J = 1 — the cramped start recovered here)")

**Exercise 12 — k-means++ probabilities.** Points $x=0,1,10$, first centroid $x=0$. The next is chosen ∝ squared distance to the nearest chosen centroid. Compute the probabilities:

In [ ]:
rest = np.array([1., 10.]); d2 = rest ** 2                # dist^2 from each to the only seed (x=0)
for xi, p in zip(rest, d2 / d2.sum()):
    print(f"P(pick x={xi:>2.0f}) = {p:.4f}")
print("-> ~99% chance of x=10: k-means++ spreads the seeds to one per natural group (expect 1/101, 100/101)")

> **Exercises 9 & 11 are pen-and-paper proofs** (the general vector derivation that $\mu_k$ is the mean, and the argument that each step can't increase $J$) — there's nothing to run; work them out and check against the reading's **Solutions**. Cell 16 and the §3 derivation give the mechanics.

## Key terms
- **Unsupervised learning** — structure from data with **no labels**; clustering is the prototype.
- **Centroid $\mu_k$** — a cluster's prototype; the **mean** of its members.
- **Assignment $r_{nk}$** — binary **hard** membership (1 iff $x_n$ in cluster $k$).
- **Distortion $J$** — $\sum_n\sum_k r_{nk}\lVert x_n-\mu_k\rVert^2$, what K-means minimises.
- **Lloyd's algorithm** — the assign→update loop; **coordinate descent** on $J$ (so $J$ only falls).
- **Local optimum** — a fixed point that's best only locally; the result depends on the start.
- **k-means++** — distance-weighted init that spreads the seeds.
- **Elbow method** — pick $K$ from the bend in the $J$-vs-$K$ curve.